In [1]:
library(forecast)
library(tseries)
library(readxl)
library(readxl)
library(tidyverse)
library(tsibble)
library(fable)
library(ggplot2)
library(lubridate)

Registered S3 method overwritten by 'quantmod':
  method            from
  as.zoo.data.frame zoo 

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Registered S3 method overwritten by 'tsibble':
  method               from 
  as_tibble.grouped_df dplyr


Attaching package: 'tsibble'


The following object is masked from 'package:lubridate':

    interval


The following objects are masked from 'package:base':

    intersect, setdiff, union


Loading required package: fabletools



In [2]:
R.version.string

[1] "R version 4.5.2 (2025-10-31 ucrt)"

In [3]:
df_tariff <- read_excel("cleandata_tariffdata_concat_tariff_rev1.xlsx", sheet = "clean tariff data")
head(df_tariff)

hts8,brief_description,mfn_text_rate,mfn_ad_val_rate,col2_text_rate,begin_effect_date,end_effective_date,Industry
<dbl>,<chr>,<chr>,<dbl>,<chr>,<dttm>,<chr>,<chr>
37040000,"Photographic plates, film, paper, paperboard and textiles, exposed but not developed",Free,0,$3.88/m2,1989-01-01,31/12/2050,Apparel
42034060,"Clothing accessories of leather or of composition leather, nesoi",Free,0,0.35,1989-01-01,31/12/2050,Apparel
53031000,"Jute and other textile bast fibers (excluding flax, true hemp and ramie), raw or retted",Free,0,Free,1989-01-01,31/12/2050,Apparel
53039000,"Jute and other textile bast fibers (excluding flax, true hemp and ramie), processed but not spun; tow and waste of these fibers",Free,0,Free,1989-01-01,31/12/2050,Apparel
53101000,Unbleached woven fabrics of jute or of other textile bast fibers of heading 5303,Free,0,0.4,1989-01-01,31/12/2050,Apparel
57050010,"Carpets and other textile floor coverings, whether or not made up, of coir, nesoi",Free,0,0.16,1989-01-01,31/12/2050,Apparel


In [4]:
df_tariff_ets <- df_tariff[, c("begin_effect_date", "mfn_ad_val_rate", "Industry")] %>%
    filter(begin_effect_date < as.Date("2025-01-01")) %>%
    mutate(mfn_ad_val_rate = mfn_ad_val_rate * 100)
head(df_tariff_ets)

begin_effect_date,mfn_ad_val_rate,Industry
<dttm>,<dbl>,<chr>
1989-01-01,0,Apparel
1989-01-01,0,Apparel
1989-01-01,0,Apparel
1989-01-01,0,Apparel
1989-01-01,0,Apparel
1989-01-01,0,Apparel


In [5]:
tail(df_tariff_ets)

begin_effect_date,mfn_ad_val_rate,Industry
<dttm>,<dbl>,<chr>
2024-01-01,26.4,Food
2024-01-01,26.4,Food
2024-01-01,26.4,Food
2024-01-01,26.4,Food
2024-01-01,26.4,Food
2024-01-01,26.4,Food


In [6]:
unique(df_tariff_ets$Industry)

[1] "Apparel"     "Electronics" "Food"

In [7]:
df_apparel = df_tariff_ets[df_tariff_ets$Industry == 'Apparel', ]
df_electronics = df_tariff_ets[df_tariff_ets$Industry == 'Electronics', ]
df_food = df_tariff_ets[df_tariff_ets$Industry == 'Food', ]

In [8]:
table(format(df_apparel$begin_effect_date, "%Y-%m"))


1989-01 1993-08 1993-11 1994-01 1995-01 1996-01 1999-01 2000-09 2000-10 2000-11 
    209       3       8     109      52      75     360       2      16       9 
2002-01 2002-10 2004-01 2007-01 2008-01 2008-12 2009-01 2012-02 2012-03 2012-10 
      9       2     726     143     112       9      18      16       6    1409 
2013-01 2014-01 2015-01 2015-07 2016-01 2016-07 2016-08 2016-12 2017-01 2017-07 
    160     238    2397       4     742      22      68     110     302      20 
2018-01 2018-04 2018-07 2018-10 2018-11 2019-01 2019-06 2019-07 2019-10 2020-01 
    248      14       4     219       6      82      16      12      14     190 
2020-04 2020-07 2020-12 2021-01 2022-01 2023-01 
     20    5128      18     294     272      72 

In [9]:
# df_apparel_mean <- tapply(df_apparel$mfn_ad_val_rate, df_apparel$begin_effect_date, mean)
df_apparel_mean <- aggregate(df_apparel$mfn_ad_val_rate, by = list(df_apparel$begin_effect_date), FUN=mean) #take mean for duplicate effective rates
names(df_apparel_mean) <- c("begin_effect_date", "mfn_ad_val_rate")
head(df_apparel_mean)

,begin_effect_date,mfn_ad_val_rate
,<dttm>,<dbl>
1,1989-01-01,2.105263
2,1993-08-11,0.000000
3,1993-11-08,0.000000
4,1994-01-01,0.000000
5,1995-01-01,0.000000
6,1996-01-01,0.000000


In [10]:
ts_apparel <- df_apparel_mean %>%
    mutate(begin_effect_date = as.Date(begin_effect_date)) %>%
    as_tsibble(index = begin_effect_date) %>%
    index_by(month = yearmonth(begin_effect_date)) %>%
    summarise(mfn_ad_val_rate = mean(mfn_ad_val_rate)) %>%
    fill_gaps() %>%
    fill(mfn_ad_val_rate, .direction = "down")

# ts_apparel <- ts_apparel %>% fill_gaps()
# ts_apparel <- ts_apparel %>% fill(mfn_ad_val_rate, .direction="down")

In [11]:
n_row_apparel <- nrow(ts_apparel)
split_apparel <- floor(0.8 * n_row_apparel)
train_apparel <- ts_apparel %>% slice(1:split_apparel)
test_apparel <- ts_apparel %>% slice((split_apparel + 1):n_row_apparel) 
ets_apparel <- train_apparel %>% model(ETS(mfn_ad_val_rate))
forecast_apparel <- ets_apparel %>% forecast(h= nrow(test_apparel))
# forecast_apparel <- ets_apparel %>% forecast(new_data = test_apparel)

In [12]:
accuracy_apparel <- accuracy(forecast_apparel, test_apparel) %>%
    mutate(Industry = "Apparel")

In [13]:
accuracy_apparel

.model,.type,ME,RMSE,MAE,MPE,MAPE,MASE,RMSSE,ACF1,Industry
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ETS(mfn_ad_val_rate),Test,-0.6953724,5.788835,4.274531,-Inf,Inf,NaN,NaN,0.6107134,Apparel


In [14]:
df_food_mean <- aggregate(df_food$mfn_ad_val_rate, by = list(df_food$begin_effect_date), FUN=mean) #take mean for duplicate effective rates
names(df_food_mean) <- c("begin_effect_date", "mfn_ad_val_rate")

ts_food <- df_food_mean %>%
    mutate(begin_effect_date = as.Date(begin_effect_date)) %>%
    as_tsibble(index = begin_effect_date) %>%
    index_by(month = yearmonth(begin_effect_date)) %>%
    summarise(mfn_ad_val_rate = mean(mfn_ad_val_rate)) %>%
    fill_gaps() %>%
    fill(mfn_ad_val_rate, .direction = "down")

n_row_food <- nrow(ts_food)
split_food <- floor(0.8 * n_row_apparel)
train_food <- ts_food %>% slice(1:split_food)
test_food <- ts_food %>% slice((split_food + 1):n_row_food) 

ets_food <- train_food %>% model(ETS(mfn_ad_val_rate))
forecast_food <- ets_food %>% forecast(h=nrow(test_food))
accuracy_food <- accuracy(forecast_food, test_food) %>%
    mutate(Industry = "Food")

In [15]:
df_electronics_mean <- aggregate(df_electronics$mfn_ad_val_rate, by = list(df_electronics$begin_effect_date), FUN=mean) #take mean for duplicate effective rates
names(df_electronics_mean) <- c("begin_effect_date", "mfn_ad_val_rate")

ts_electronics <- df_electronics_mean %>%
    mutate(begin_effect_date = as.Date(begin_effect_date)) %>%
    as_tsibble(index = begin_effect_date) %>%
    index_by(month = yearmonth(begin_effect_date)) %>%
    summarise(mfn_ad_val_rate = mean(mfn_ad_val_rate)) %>%
    fill_gaps() %>%
    fill(mfn_ad_val_rate, .direction = "down")

n_row_electronics <- nrow(ts_electronics)
split_electronics <- floor(0.8 * n_row_electronics)
train_electronics <- ts_electronics %>% slice(1:split_electronics)
test_electronics <- ts_electronics %>% slice((split_electronics + 1):n_row_electronics) 

ets_electronics <- train_electronics %>% model(ETS(mfn_ad_val_rate))
forecast_electronics <- ets_electronics %>% forecast(h=nrow(test_electronics))
accuracy_electronics <- accuracy(forecast_electronics, test_electronics) %>%
    mutate(Industry = "Electronics")

In [16]:
accuracy_summary <- bind_rows(accuracy_apparel, accuracy_food, accuracy_electronics) %>%
    select(Industry, RMSE, MAE)
accuracy_summary

Industry,RMSE,MAE
<chr>,<dbl>,<dbl>
Apparel,5.788835,4.274531
Food,1.805614,1.239282
Electronics,1.862831,1.557192
